# Red Neuronal (MLP) - MIAX14 Hackathon

Modelo simple de red neuronal para los índices **A, B, D, E** (C y F siguen bloqueados de la submission v1).

**Idea clave**: los árboles (LightGBM/XGBoost) no extrapolan. Una MLP entrenada sobre **log-retornos**
(target estacionario) sí puede seguir subiendo más allá del máximo histórico, porque predice un retorno
diario pequeño y el nivel se reconstruye por composición: `nivel(t) = nivel(t-1) · exp(r̂(t))`.

**Estabilidad**: en modo autorregresivo, las features de *nivel* (lags, medias móviles) divergen y hacen
explotar la red. Por eso la MLP usa solo features **estacionarias** (retornos + exógenas) y un clip duro
del log-retorno (±10%/día).

In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from features import load_data, build_feature_matrix, INDICES
from predict import _build_row, predict_autoregressive
from neural_model import NeuralReturnModel, select_stationary_features
import lightgbm as lgb

sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (14, 5)

## 1. Cargar datos y features

In [ ]:
data = load_data('../data')
train_feat, test_feat = build_feature_matrix(data, data_dir='../data')
feature_cols = [c for c in train_feat.columns if c not in INDICES]

stat_cols = select_stationary_features(feature_cols)
print(f'Features totales: {len(feature_cols)}')
print(f'Features estacionarias (para la NN): {len(stat_cols)}')
print('Ejemplos:', stat_cols[:8])

## 2. Validación autorregresiva: NN vs Árboles
Dejamos el último año fuera y comparamos en modo autorregresivo (252 pasos).

In [ ]:
VAL = 252
levels = train_feat[INDICES]
X_tr = train_feat[feature_cols].iloc[:-VAL]
tf_val = train_feat.iloc[-VAL:]
NN_IDX = ['Index_A', 'Index_B', 'Index_D', 'Index_E']

# NN sobre log-retornos
nn = NeuralReturnModel(indices=NN_IDX)
nn.fit(X_tr, levels.iloc[:-VAL])
nn.feature_cols = feature_cols

# Árboles sobre niveles
tree = {}
for idx in INDICES:
    m = lgb.LGBMRegressor(n_estimators=700, learning_rate=0.05, num_leaves=127, verbose=-1)
    m.fit(X_tr, levels[idx].iloc[:-VAL]); tree[idx] = m

In [ ]:
def autoreg(use_nn_for):
    history = levels.iloc[:-VAL].copy()
    ve = tf_val[feature_cols]
    out = {idx: [] for idx in INDICES}
    for date in tf_val.index:
        row = _build_row(history, date, ve.loc[date], feature_cols)
        last = {idx: history[idx].iloc[-1] for idx in INDICES}
        nn_pred = nn.predict(row, last_levels=last) if use_nn_for else None
        new = {}
        for idx in INDICES:
            if use_nn_for and idx in use_nn_for:
                new[idx] = float(nn_pred[idx].iloc[0])
            else:
                new[idx] = float(tree[idx].predict(row)[0])
        for idx in INDICES: out[idx].append(new[idx])
        history = pd.concat([history, pd.DataFrame([new], index=[date])])
    return pd.DataFrame(out, index=tf_val.index)

real   = levels.iloc[-VAL:]
p_tree = autoreg(use_nn_for=None)
p_nn   = autoreg(use_nn_for=NN_IDX)

In [ ]:
rows = []
for col in NN_IDX:
    rt = np.sqrt(((p_tree[col]-real[col])**2).mean())
    rn = np.sqrt(((p_nn[col]-real[col])**2).mean())
    rows.append({'Índice': col, 'RMSE_Árbol': rt, 'RMSE_NN': rn,
                 'Ganador': 'NN' if rn < rt else 'Árbol',
                 'Mejora_%': (1 - rn/rt)*100})
pd.DataFrame(rows).round(1)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 9))
for ax, col in zip(axes.flatten(), NN_IDX):
    real[col].plot(ax=ax, color='black', lw=1.2, label='Real')
    p_tree[col].plot(ax=ax, color='steelblue', lw=1, ls='--', label='Árbol')
    p_nn[col].plot(ax=ax, color='crimson', lw=1, ls='--', label='NN')
    ax.set_title(col); ax.legend()
plt.suptitle('Validación autorregresiva (último año): Real vs Árbol vs NN', y=1.01)
plt.tight_layout(); plt.show()

## 3. Modelo final + network anchor para A

Estrategia validada:
- **NN para A y B** (gana a los árboles en autorregresivo).
- **Index_D = ghost(A)**: `D(t) ≈ 1.0001815·A(t-1) + 3.116` (R²=0.99999), se deriva de A.
- **Árbol para E** (gana a la NN).
- **Network anchor para A**: las network metrics correlacionan ~0.90 con A *en niveles* y sus
  valores en el test son conocidos (futuro). Blend **80% NN + 20% anchor** bajó el RMSE de A de
  101k → 91k en validación. Como las network suben +90% en el test, anclan A al alza con fundamento.
- **C y F** bloqueados de la submission v1.

El modelo final se entrena con TODA la serie (incluida la validación).

In [ ]:
from models import EnsembleModel
from features import INDEX_D_COEF, INDEX_D_INTERCEPT
from network_anchor import network_anchor_series, blend_index_a_with_anchor

X_full = train_feat[feature_cols]

# NN final para A y B (D se deriva de A via ghost)
nn_final = NeuralReturnModel(indices=['Index_A', 'Index_B'])
nn_final.fit(X_full, levels)
nn_final.feature_cols = feature_cols

# Ensemble de árboles para E (y C/F que se bloquean luego)
tree_final = EnsembleModel(INDICES, index_d_ghost_weight=0.7, lgbm_weight=0.5)
tree_final.lgbm.fit(X_full, levels)
tree_final.xgb.fit(X_full, levels)
tree_final.feature_cols = feature_cols

In [ ]:
# Autorregresivo híbrido: NN para A,B ; ghost para D ; árbol para C,E,F
history = data['train_idx'][INDICES].copy()
out = {idx: [] for idx in INDICES}
from tqdm.notebook import tqdm
for date in tqdm(test_feat.index):
    row = _build_row(history, date, test_feat.loc[date], feature_cols)
    last = {idx: history[idx].iloc[-1] for idx in INDICES}
    nn_pred = nn_final.predict(row, last_levels=last)
    tree_pred = tree_final.predict(row, index_a_lag1=pd.Series([last['Index_A']], index=[date]))
    new = {}
    new['Index_A'] = float(nn_pred['Index_A'].iloc[0])
    new['Index_B'] = float(nn_pred['Index_B'].iloc[0])
    new['Index_D'] = INDEX_D_COEF * last['Index_A'] + INDEX_D_INTERCEPT  # ghost: D(t)=A(t-1)
    new['Index_C'] = float(tree_pred['Index_C'].iloc[0])
    new['Index_E'] = float(tree_pred['Index_E'].iloc[0])
    new['Index_F'] = float(tree_pred['Index_F'].iloc[0])
    for idx in INDICES: out[idx].append(new[idx])
    history = pd.concat([history, pd.DataFrame([new], index=[date])])

test_preds = pd.DataFrame(out, index=test_feat.index)

In [ ]:
# Network anchor para A (blend 80/20), luego re-derivar D del A corregido
net_te = pd.read_csv('../data/test_network_metrics.csv', parse_dates=['Date'], index_col='Date')
anchor_A = network_anchor_series(data['train_idx'], data['train_net'], net_te, test_preds.index, 'Index_A')
test_preds['Index_A'] = blend_index_a_with_anchor(test_preds['Index_A'], anchor_A)

# D = ghost(A corregida)
A_shift = test_preds['Index_A'].shift(1)
A_shift.iloc[0] = data['train_idx']['Index_A'].iloc[-1]
test_preds['Index_D'] = INDEX_D_COEF * A_shift + INDEX_D_INTERCEPT

# Bloquear C y F de la submission v1
locked = pd.read_csv('../results/locked_predictions_C_F.csv', parse_dates=['Date']).set_index('Date')
locked = locked.rename(columns={'pred_index_c': 'Index_C', 'pred_index_f': 'Index_F'})
for idx in ['Index_C', 'Index_F']:
    test_preds[idx] = locked[idx].reindex(test_preds.index).values

# Resumen de cambios en el horizonte
for col in INDICES:
    last = data['train_idx'][col].iloc[-1]
    chg = (test_preds[col].iloc[-1]/last - 1)*100
    print(f'  {col}: {last:>12,.0f} → {test_preds[col].iloc[-1]:>12,.0f}  ({chg:+.1f}%)')

## 4. Guardar submission con formato del template

In [ ]:
import os
COLS = {f'Index_{c}': f'pred_index_{c.lower()}' for c in 'ABCDEF'}
out_df = test_preds.rename(columns=COLS)
out_df.index.name = 'Date'
out_df = out_df.reset_index()
out_df['Date'] = pd.to_datetime(out_df['Date']).dt.strftime('%Y-%m-%d')
out_df = out_df[['Date'] + list(COLS.values())]

os.makedirs('../predicciones', exist_ok=True)
path = '../predicciones/submission4_neural_network.xlsx'
out_df.to_excel(path, index=False, sheet_name='submission')
print(f'Guardado: {path}')
out_df.head()

## 5. Plot final: histórico (negro) + predicción NN (naranja)

In [ ]:
import matplotlib.dates as mdates
hist = data['train_idx']
fig, axes = plt.subplots(3, 2, figsize=(18, 14))
for ax, col in zip(axes.flatten(), INDICES):
    h = hist[col].iloc[-500:]; p = test_preds[col]
    ax.plot(h.index, h.values, color='black', lw=0.9, label='Histórico')
    ax.plot([h.index[-1], p.index[0]], [h.iloc[-1], p.iloc[0]], color='darkorange', lw=1.1)
    ax.plot(p.index, p.values, color='darkorange', lw=1.4, label='Predicción')
    ax.axvline(p.index[0], color='gray', ls='--', lw=0.8, alpha=0.7)
    chg = (p.iloc[-1]/h.iloc[-1]-1)*100
    ax.set_title(f'{col}  ({chg:+.1f}%)'); ax.legend(fontsize=8)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{x:,.0f}'))
plt.suptitle('Submission 4 (NN para A/B/D) — histórico vs predicción', y=1.01)
plt.tight_layout()
plt.savefig('../predicciones/submission4_neural.png', dpi=150, bbox_inches='tight')
plt.show()